# Code to run long simulations of networking with eq. and prod. phase

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import readdy
import math
import scipy
import pandas as pd
import sklearn
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import RobustScaler
import contextlib
import io
import pickle
import torch
from botorch.models import SingleTaskGP
from botorch.models.transforms import Normalize, Standardize
from botorch.fit import fit_gpytorch_mll
from gpytorch.mlls import ExactMarginalLogLikelihood
from gpytorch.likelihoods import FixedNoiseGaussianLikelihood
from botorch.acquisition import LogExpectedImprovement
from botorch.acquisition import ExpectedImprovement
from botorch.acquisition import UpperConfidenceBound
from botorch.optim import optimize_acqf
import multiprocessing

from scipy.optimize import curve_fit
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_percentage_error,mean_squared_log_error,mean_absolute_error
import threading
from concurrent.futures import ThreadPoolExecutor
from scipy.interpolate import interp1d

import sys
import os
import seaborn as sns
#sys.path.append(os.path.abspath(os.path.join('..', 'python_scripts')))

from NCG_readdy_sim_4 import Debye_length, Debye_Hueckel_pot_rep_strength ,Diff_at23C,visc_T,Diff_ES

from NCG_readdy_sim_4 import run_sim_3,tcl_file_gen_3,all_types

from NCG_Bay_opt_readdy_sim_3 import run_p,run_p_dict,gt_1_3,gt_1_1,gt_1_2,gt_1_4, cg_angle_distr_c1_4_ba_aa_oa_la,cg_angle_distr_c1_1_hp,cg_angle_distr_c1_1_bm, eval_sim_c1_1_la,eval_sim_c1_1_hp,eval_sim_c1_1_aa,eval_sim_c1_2_ba_oa, eval_sim_c1_4_ba_aa_oa_la,eval_sim_c1_1_bm,apply_gp, run_gp_11

from NCG_Bay_opt_readdy_sim_3 import gen_all_comb_from_list_of_lists_2

from NCG_readdy_sim_eval_1 import get_particle_number_from_sims
from NCG_readdy_sim_eval_2 import get_time_avg_storage_loss_mod_2, get_subgraph_eigenvalues, relax_times, storage_mod, loss_mod_deref,relaxation_mod,get_time_avg_storage_loss_mod_4,get_subgraph_eigenvalues,get_time_avg_storage_loss_mod

# Calculating complex shear modulus using extension of Rouse-Zimm theory

X-motif design used in Corad et al Increasing valence pushes DNA nanostar networks to the isostatic point

# Units and temperature

In [ ]:
#Set units used in ReaDDy and simulation temperature
custom_units = {'length_unit':'nanometer',
                'time_unit': 'nanoseconds',
                'energy_unit': 'joule'}
system_temperature=273.15 + 20  #293
kbt=system_temperature*1.380649*10**-23

# Diffusion constant

In [ ]:
#Diffusion constant either based on Einstein-Smoluchowski, or on exp results for oligos at 23°C
# viscositiy: https://srd.nist.gov/jpcrdreprint/1.555581.pdf
# scaling diffusion constant with T: https://en.wikipedia.org/wiki/Einstein_relation_(kinetic_theory)
# Diffusion constant DNA oligos at 23°C: https://www.sciencedirect.com/science/article/pii/S0021925818310871


#Diffusion const for BP of ds DNA (known at 23°C)
Diff_at23C_1=Diff_at23C(8)

#Viscosity water at sim temp and 23°C
visc_T_sys=visc_T(system_temperature)
visc_T_23=visc_T(273.15+23)

#rescale diffusion const to sim temperature and viscosity (assume scaling like Einstein-Smoluchowski D= kb*T/(6*pi*r*visc) )
Diff_sys_temp=Diff_at23C_1*system_temperature/(23+273.15)  *visc_T_23/visc_T_sys       

#use Einstein-Smoluchowski for sphere
Diff_sys_temp_ES=Diff_ES(T=system_temperature,r=2.72/2 *10**-9,eta=visc_T_sys)*10**9 
Diff_T_23_ES=Diff_ES(T=23+273.15,r=2.72/2 *10**-9,eta=visc_T_sys)*10**9


print("D for 8bp at 23°C /",23+273.15,"K =",Diff_at23C_1,"nm^2/ns") 
print("D for sphere with r=2.72/2 nm at 23°C /",23+273.15,"K =", Diff_T_23_ES,"nm^2/ns")
print("############################################")
print("mu at",system_temperature-273.15,"°C /", system_temperature,"K =",visc_T_sys," Pa*s" )
print("mu at","23°C /", 23+273.15,"K =" ,visc_T_23," Pa*s" )
print("############################################")
print("D for 8bp at",system_temperature-273.15,"°C /",system_temperature,"K =",Diff_sys_temp,"nm^2/ns")
print("D for sphere with r=2.72/2 nm at" ,system_temperature-273.15,"°C /",system_temperature,"K =",Diff_sys_temp_ES,"nm^2/ns")


# Fixed model parameters

In [ ]:
###################
#fixed params:
###################

#bond potential
k_bond_ds=2*10**-19
k_bond_ss=1*10**-19

#bond eq. distance
eq_dist_bb=2.72 

#bond eq. distance fused sticky-ends
eq_dist_fused_bb=2/3 *2.72
eq_dist_bb_s=2/3 *2.72 # distance arm-stickyend

#bond potential surf
k_angle_ss_surf=2e-20  #2.8666666666666663e-20     #2*10**-21
k_angle_ds_surf=3.6666666666666664e-20     #2*10**-20

#angle between monomers V=k*(theta-theta_0)^2
theta_0=np.pi
theta_90=np.pi/2

theta_60=np.pi *1/3
theta_120=np.pi * 2/3
theta_270=np.pi * 3/2
theta_base_y=np.pi*2/3

#volume exclusion
#har rep
k_ex_ds_ds=2*10**-20  *5
k_ex_ss_ds=8*10**-21  *5
k_ex_ss_ss=4*10**-21  *5
dist_ex=1.5          *1 

#DH
c_NaCl = 150 # NaCl concentration in milli mol/L
L_1=6.022*10**23 *c_NaCl * (1.602*10**-19)**2 

charge_eff=0.815 #https://pubs.aip.org/aip/jcp/article/142/23/234901/193554/Introducing-improved-structural-properties-and
DH_len=Debye_length(L=L_1,T=system_temperature) *10**9 #for nm
DH_strength_ds_ds=Debye_Hueckel_pot_rep_strength(q1=charge_eff*16,q2=charge_eff*16,T=system_temperature)*10**9 #for nm
DH_strength_ss_ds=Debye_Hueckel_pot_rep_strength(q1=charge_eff*8,q2=charge_eff*16,T=system_temperature)*10**9 #for nm
DH_strength_ss_ss=Debye_Hueckel_pot_rep_strength(q1=charge_eff*8,q2=charge_eff*8,T=system_temperature)*10**9 #for nm
DH_cutoff=DH_len*4

volume_excl_type="DH"

print("Debye length at",c_NaCl,"mM NaCl:", DH_len, "nm", "cutoff at", DH_cutoff, "nm")
print("DH pot strength ds-ds:", DH_strength_ds_ds, "J*nm")
print("DH pot strength ss-ds:", DH_strength_ss_ds, "J*nm")
print("DH pot strength ss-ss:", DH_strength_ss_ss, "J*nm")
print("Energy of rep. ds-ds at 1 nm:",DH_strength_ds_ds/1 *np.exp(-1/DH_len),"J" )
#DH_strength_ds_ds=DH_strength_ds_ds
#DH_strength_ss_ds=DH_strength_ss_ds
#DH_strength_ss_ss=DH_strength_ss_ss
#DH_cutoff=DH_cutoff

#probabilities bound/unbound
p_SE_fused_x=99.8
p_SE_fused_ortho_x=99.8

p_SE_HP_x=83.8
p_SE_HP_ortho_x=59.9

p_surf_HP_x=6.6
p_surf_HP_ortho_x=6.6

p_SE_fused_x_c=89.1
p_SE_HP_x_c=24.2


#binding/unbinding rates
#rate of selecting fission reaction in sim
k_fiss_rate_sel=10**7
#y motif
k_b_y_1=0.05
k_ub_y_1=0.001

#y motif hp
k_b_y_hp_1=0
k_ub_y_hp_1=0

#############

#x motif 1
k_b_x_1=0     #0.05
k_ub_x_1=0    #k_b_x_1 * (100-p_SE_fused_x)/p_SE_fused_x  #0.001

#x motif 1 surface
k_b_x_surf_1=k_b_x_1 
k_ub_x_surf_1=k_ub_x_1

#x motif 1 hp
k_b_x_hp_1=0      #0.005
k_ub_x_hp_1=0      #k_b_x_hp_1 * (100-p_SE_HP_x)/p_SE_HP_x 

#x motif 1 surf hp
k_b_x_surf_hp_1=0     #0.005
k_ub_x_surf_hp_1=0     #k_b_x_surf_hp_1 * (100-p_surf_HP_x)/p_surf_HP_x

############

#x motif 2
k_b_x_2=0
k_ub_x_2=0

#x motif 2 surface
k_b_x_surf_2=0 
k_ub_x_surf_2=0

#x motif 2 hp
k_b_x_hp_2=0
k_ub_x_hp_2=0

#x motif 2 surf hp
k_b_x_surf_hp_2=0
k_ub_x_surf_hp_2=0

############

#x motif conrad

#x motif 1
k_b_x_c1=0.05    #0.05
k_ub_x_c1=0 #4.017716642881403e-05  #0.000005#0.0005 #0.0000076   #k_b_x_c1 * (100-p_SE_fused_x_c)/p_SE_fused_x_c  #0.001

#x motif 1 hp
k_b_x_hp_c1=0.005
k_ub_x_hp_c1=k_b_x_hp_c1 * (100-p_SE_HP_x_c)/p_SE_HP_x_c 


#time step
t_step=10*10**-3  #standard: 10*10**-3 #appears to work with up to 25*10**-3

#reaction scale
rea_s=2.7
rea_c1_s=2.7


In [ ]:
(5e-05)-0.000005

# Previously optimized parameters

In [ ]:
#optimal conrad x motif params

with open('/home/aaron/phd/Nanomotif_Coarse_Graining_1/ML_parameter_opt_1/NCG_X_motif_conrad_1/optimal_params_1.pkl', 'rb') as file:
    dict_opt_params_x_motif = pickle.load(file)

k_angle_ds_base=dict_opt_params_x_motif["k_angle_ds_base"] 
k_angle_ds_arm=dict_opt_params_x_motif["k_angle_ds_arm"]
k_angle_ds_oppa=dict_opt_params_x_motif["k_angle_ds_oppa"]
k_angle_ss_link=dict_opt_params_x_motif["k_angle_ss_link"]
k_angle_ds_fused_link=dict_opt_params_x_motif["k_angle_ds_fused_link"]
k_angle_ds_hp=dict_opt_params_x_motif["k_angle_ds_hp"]
theta_base_x=dict_opt_params_x_motif["theta_base_x"]
theta_oppa_x=dict_opt_params_x_motif["theta_oppa_x"]

#optimal ss surf paramters
with open('ML_parameter_opt_1/NCG_X_4_surf_1/optimal_ss_surf_params_2.pkl', 'rb') as file:
    dict_opt_params_surf = pickle.load(file)
k_angle_ss_surf=dict_opt_params_surf["k_angle_ss_surf"]

In [ ]:
print(dict_opt_params_x_motif)

# Run simulations

# General simulation parameters

In [ ]:
#general simulation parameters
system_temperature=system_temperature
box_size=[69.718,69.718,69.718]
custom_units=custom_units
periodic_bc_x=True
periodic_bc_y=True
periodic_bc_z=True

r_sp=15 #radius of spherical inclusion pot
sp_incl_orig=[0,0,0] #origin of sperical inclusion pot
k_sp=0 #k of spherical inclusion, if zero no spherical inclusion
box_a=[-60,-60,-60] #offset of box potential from centre
box_b=[120,120,120] #total length of box potential
k_box=0 #1 *10**-20 #k of box potential


#number of particles
num_y_motif_1=0
num_x_motif_1=0
num_x_motif_2=0
num_x_motif_c1=100
num_surface_1=0

#surface section lengths
num_rep_unit=1
len_fill_surf_a=2
len_binding_surf=4
len_fill_surf_b=2


#option to use HP template
use_y_1_HP_template=False
use_x_1_HP_template=False
use_x_2_HP_template=False
use_x_c1_HP_template=False

#positions of particles
y_motif_1_ini=[np.asarray([[0,0,0]])]
y_motif_1_rot=False

x_motif_1_ini=[np.asarray([[0,0,0]])]
x_motif_1_rot=False

x_motif_2_ini=[np.asarray([[-10,0,0]])]
x_motif_2_rot=False

x_motif_c1_ini=[np.asarray([e]) for e in np.random.uniform(size=(num_x_motif_c1,3)) * np.asarray(box_size)*0.8 - np.asarray(box_size)*0.8 * 0.5]  #[np.asarray([[0,0,0]])] #[np.asarray([[0,0,0]])]
x_motif_c1_rot=True

surf_1_ini=[np.asarray([[-40,-40,-40]])]

#time steps
t_tot=3*10**7  #500000

#evaluate reactions


#folder
folder="/home/aaron/phd/Nanomotif_Coarse_Graining_1/ML_parameter_opt_1/NCG_X_motif_conrad_1/shear_mod_test_1/"
name="x_test_1_"
add_num_1=0
     

#other options

sim_kernel="SingleCPU"
sim_threads=1
track_num_particles=["y_link_a_1","y_link_fused_a_1","y_link_b_1","y_link_fused_b_1","y_link_c_1","y_link_fused_c_1",
                                                                   "x_link_a_1","x_link_fused_a_1","x_link_b_1","x_link_fused_b_1","x_link_c_1","x_link_fused_c_1","x_surf_link_d_1","x_surf_link_fused_d_1",
                                                                    "x_link_a_2","x_link_fused_a_2","x_link_b_2","x_link_fused_b_2","x_link_c_2","x_link_fused_c_2","x_surf_link_d_2","x_surf_link_fused_d_2",
                                                                    'x_link_hp_a_1', 'x_link_hp_b_1', 'x_link_hp_c_1',"x_surf_link_hp_d_1",
                                                                    'x_link_hp_a_2', 'x_link_hp_b_2', 'x_link_hp_c_2',"x_surf_link_hp_d_2",
                                                                    "x_link_a_c1","x_link_fused_a_c1","x_link_b_c1","x_link_fused_b_c1","x_link_c_c1","x_link_fused_c_c1","x_link_d_c1","x_link_fused_d_c1",
                                                                    'x_link_hp_a_c1', 'x_link_hp_b_c1', 'x_link_hp_c_c1',"x_link_hp_d_c1",
                                                                   ]
eval_reac=True
eval_top_reac=True

checkpoint_path=None #"ML_parameter_opt_1/NCG_test_1/"

new_checkpoint=None
max_n_saves=1 #max checkpoint saves
rec_stride=5000 #100 #recording stride
sim_skin=0 #1000 #simulation skin






In [ ]:
#input dictionary
run_sim_1_input={"name":name,
             "folder":folder,
             "add_num_1":add_num_1,
              
             "system_temperature":system_temperature,
             "box_size":box_size,
             "custom_units":custom_units,
             "periodic_bc_x":periodic_bc_x,
             "periodic_bc_y":periodic_bc_y,
             "periodic_bc_z":periodic_bc_z,
              
             "r_sp":r_sp,
             "sp_incl_orig":sp_incl_orig,
             "k_sp":k_sp,
             "box_a":box_a,
             "box_b":box_b,
             "k_box":k_box,
    
              
             "num_y_motif_1":num_y_motif_1,
             "num_x_motif_1":num_x_motif_1,
             "num_x_motif_2":num_x_motif_2,
             "num_x_motif_c1":num_x_motif_c1,
             "num_surface_1":num_surface_1,
             "num_rep_unit":num_rep_unit,
             "len_fill_surf_a":len_fill_surf_a,
             "len_binding_surf":len_binding_surf,
             "len_fill_surf_b":len_fill_surf_b,
             "y_motif_1_ini":y_motif_1_ini,
             "x_motif_1_ini":x_motif_1_ini,
             "x_motif_2_ini":x_motif_2_ini,
             "x_motif_c1_ini":x_motif_c1_ini,
             "surf_1_ini":surf_1_ini,
             "y_motif_1_rot":y_motif_1_rot,
             "x_motif_1_rot":x_motif_1_rot,
             "x_motif_2_rot":x_motif_2_rot,
             "x_motif_c1_rot":x_motif_c1_rot,
                 
             "use_y_1_HP_template":use_y_1_HP_template,
             "use_x_1_HP_template":use_x_1_HP_template,
             "use_x_2_HP_template":use_x_2_HP_template,
             "use_x_c1_HP_template":use_x_c1_HP_template,
                 
             "Diff_sys_temp":Diff_sys_temp,
             "k_bond_ds":k_bond_ds,
             "k_bond_ss":k_bond_ss,
             "eq_dist_bb":eq_dist_bb,
             "eq_dist_fused_bb":eq_dist_fused_bb,
             "eq_dist_bb_s":eq_dist_bb_s,
             "k_angle_ds_base":k_angle_ds_base,
             "k_angle_ds_arm":k_angle_ds_arm,
             "k_angle_ds_oppa":k_angle_ds_oppa,
             "k_angle_ss_link":k_angle_ss_link,
             "k_angle_ds_fused_link":k_angle_ds_fused_link,
             "k_angle_ds_hp":k_angle_ds_hp,
             "k_angle_ss_surf":k_angle_ss_surf,
             "k_angle_ds_surf":k_angle_ds_surf,
             "theta_0":theta_0,
             "theta_90":theta_90,
             "theta_60":theta_60,
             "theta_120":theta_120,
             "theta_270":theta_270,
             "theta_base_x":theta_base_x,
             "theta_base_y":theta_base_y,
             "theta_oppa_x":theta_oppa_x,
             
             "volume_excl_type":volume_excl_type,
             "k_ex_ds_ds":k_ex_ds_ds,
             "k_ex_ss_ds":k_ex_ss_ds,
             "k_ex_ss_ss":k_ex_ss_ss,
             "dist_ex":dist_ex,
             "DH_strength_ds_ds":DH_strength_ds_ds,
             "DH_strength_ss_ds":DH_strength_ss_ds,
             "DH_strength_ss_ss":DH_strength_ss_ss,
             "DH_len":DH_len,
             "DH_cutoff":DH_cutoff,
              
              
             "k_fiss_rate_sel":k_fiss_rate_sel,
             "k_b_y_1":k_b_y_1,
             "k_ub_y_1":k_ub_y_1,
             "k_b_y_hp_1":k_b_y_hp_1,
             "k_ub_y_hp_1":k_ub_y_hp_1,
             "k_b_x_1":k_b_x_1,
             "k_ub_x_1":k_ub_x_1,
             "k_b_x_surf_1":k_b_x_surf_1,
             "k_ub_x_surf_1":k_ub_x_surf_1,
             "k_b_x_hp_1":k_b_x_hp_1,
             "k_ub_x_hp_1":k_ub_x_hp_1,
             "k_b_x_surf_hp_1":k_b_x_surf_hp_1,
             "k_ub_x_surf_hp_1":k_ub_x_surf_hp_1,
             "k_b_x_2":k_b_x_2,
             "k_ub_x_2":k_ub_x_2,
             "k_b_x_surf_2":k_b_x_surf_2,
             "k_ub_x_surf_2":k_ub_x_surf_2,
             "k_b_x_hp_2":k_b_x_hp_2,
             "k_ub_x_hp_2":k_ub_x_hp_2,
             "k_b_x_surf_hp_2":k_b_x_surf_hp_2,
             "k_ub_x_surf_hp_2":k_ub_x_surf_hp_2,
             "k_b_x_c1":k_b_x_c1,
             "k_ub_x_c1":k_ub_x_c1,
             "k_b_x_hp_c1":k_b_x_hp_c1,
             "k_ub_x_hp_c1":k_ub_x_hp_c1,
                 
             "t_step":t_step,
             "rea_s":rea_s,
             "rea_c1_s":rea_c1_s,
             "t_tot":t_tot,
             
             "sim_kernel":sim_kernel,
             "sim_threads":sim_threads,
             "track_num_particles":track_num_particles,
             "eval_reac":eval_reac,
             "eval_top_reac":eval_top_reac,
             "checkpoint_path":checkpoint_path,
             "new_checkpoint":new_checkpoint,
             "max_n_saves":max_n_saves,
             "rec_stride":rec_stride,
             "sim_skin":sim_skin
    
    
}

# Simulation runs

In [ ]:
## run 1

In [ ]:
%%capture
#create equilibration checkpoints


#run multiple copys of sim in parallel
gen_sim_list=[]
gen_sim_param_list=[]

for i in range(5):
    gen_sim_list.append(run_sim_3)
    run_sim_1_input_c=run_sim_1_input.copy()
    run_sim_1_input_c["t_tot"]=2*10**6
    run_sim_1_input_c["k_b_x_c1"]=0    
    run_sim_1_input_c["k_ub_x_c1"]=0  
    run_sim_1_input_c["k_b_x_hp_c1"]=0
    run_sim_1_input_c["k_ub_x_hp_c1"]=0
    run_sim_1_input_c["new_checkpoint"]="/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_shear_mod_2/eq_checkpoint_"+str(i)+"/"
    #run_sim_1_input_c["checkpoint_path"]= "/home/aaron/phd/Nanomotif_Coarse_Graining_1/ML_parameter_opt_1/NCG_X_motif_conrad_1/shear_mod_test_1/checkpoints_15/"
 
    run_sim_1_input_c["folder"]="/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_shear_mod_2/eq_checkpoint_"+str(i)+"/"
    run_sim_1_input_c["add_num_1"]=i
    run_sim_1_input_c["name"]="x_c_mech_opt_eq_"
    gen_sim_param_list.append(run_sim_1_input_c)
    
run_p_dict(gen_sim_list,gen_sim_param_list)


In [ ]:
%%capture
#run multiple copys of sim in parallel
gen_sim_list=[]
gen_sim_param_list=[]

for i in range(5):
    gen_sim_list.append(run_sim_3)
    run_sim_1_input_c=run_sim_1_input.copy()
    run_sim_1_input_c["t_tot"]=int(1.5*10**7)
    run_sim_1_input_c["k_ub_x_c1"]=0
    run_sim_1_input_c["new_checkpoint"]="/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_shear_mod_2/prod_checkpoint_"+str(i)+"/"
    run_sim_1_input_c["checkpoint_path"]="/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_shear_mod_2/eq_checkpoint_"+str(i)+"/"
 
    run_sim_1_input_c["folder"]="/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_shear_mod_2/prod_checkpoint_"+str(i)+"/"
    run_sim_1_input_c["add_num_1"]=i
    run_sim_1_input_c["name"]="x_c_mech_opt_prod_"
    gen_sim_param_list.append(run_sim_1_input_c)
    
run_p_dict(gen_sim_list,gen_sim_param_list)

In [ ]:
%%capture
#run multiple copys of sim in parallel
gen_sim_list=[]
gen_sim_param_list=[]

for i in range(5):
    folder_ch="/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_shear_mod_2/prod_checkpoint_"+str(i)+"/"
    folder_pr="/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_shear_mod_2/prod_cont_checkpoint_"+str(i)+"/"
    gen_sim_list.append(run_sim_3)
    run_sim_1_input_c=run_sim_1_input.copy()
    run_sim_1_input_c["t_tot"]=int(1.5*10**7)
    run_sim_1_input_c["k_ub_x_c1"]=0
    run_sim_1_input_c["new_checkpoint"]=folder_pr
    run_sim_1_input_c["checkpoint_path"]=folder_ch
 
    run_sim_1_input_c["folder"]=folder_pr
    run_sim_1_input_c["add_num_1"]=i
    run_sim_1_input_c["name"]="x_c_mech_opt_prod_cont_"
    gen_sim_param_list.append(run_sim_1_input_c)
    
run_p_dict(gen_sim_list,gen_sim_param_list)

In [ ]:
%%capture
#run multiple copys of sim in parallel
gen_sim_list=[]
gen_sim_param_list=[]

for i in range(5):
    folder_ch="/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_shear_mod_2/prod_cont_checkpoint_"+str(i)+"/"
    folder_pr="/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_shear_mod_2/prod_cont_2_checkpoint_"+str(i)+"/"
    os.makedirs(folder_pr)
    gen_sim_list.append(run_sim_3)
    run_sim_1_input_c=run_sim_1_input.copy()
    run_sim_1_input_c["t_tot"]=int(3*10**7)
    run_sim_1_input_c["k_ub_x_c1"]=0
    run_sim_1_input_c["new_checkpoint"]=folder_pr
    run_sim_1_input_c["checkpoint_path"]=folder_ch
 
    run_sim_1_input_c["folder"]=folder_pr
    run_sim_1_input_c["add_num_1"]=i
    run_sim_1_input_c["name"]="x_c_mech_opt_prod_cont_2_"
    gen_sim_param_list.append(run_sim_1_input_c)
    
run_p_dict(gen_sim_list,gen_sim_param_list)

# Simulation run 2

In [ ]:
%%capture
#create equilibration checkpoints


#run multiple copys of sim in parallel
gen_sim_list=[]
gen_sim_param_list=[]

for i in range(5):
    folder_ch="/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_shear_mod_inv_3_fast/eq_checkpoint_"+str(i)+"/"
    folder_pr="/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_shear_mod_inv_3_fast/prod_checkpoint_"+str(i)+"/"
    os.makedirs(folder_ch)
    os.makedirs(folder_pr)
    gen_sim_list.append(run_sim_3)
    run_sim_1_input_c=run_sim_1_input.copy()
    run_sim_1_input_c["t_tot"]=2*10**6
    run_sim_1_input_c["k_b_x_c1"]=0    
    run_sim_1_input_c["k_ub_x_c1"]=0  
    run_sim_1_input_c["k_b_x_hp_c1"]=0
    run_sim_1_input_c["k_ub_x_hp_c1"]=0
    run_sim_1_input_c["new_checkpoint"]=folder_ch
    #run_sim_1_input_c["checkpoint_path"]= "/home/aaron/phd/Nanomotif_Coarse_Graining_1/ML_parameter_opt_1/NCG_X_motif_conrad_1/shear_mod_test_1/checkpoints_15/"
 
    run_sim_1_input_c["folder"]=folder_ch
    run_sim_1_input_c["add_num_1"]=i
    run_sim_1_input_c["name"]="x_c_mech_opt_eq_"
    gen_sim_param_list.append(run_sim_1_input_c)
    
run_p_dict(gen_sim_list,gen_sim_param_list)


In [ ]:
traj_mech3=readdy.Trajectory("/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_shear_mod_inv_3_fast/prod_checkpoint_0/x_c_mech_opt_prod_0.h5")
tcl_file_mech=tcl_file_gen_3(traj_file="x_c_mech_opt_prod_0.h5.xyz",box_dim_s="69.718 69.718 69.718")
np.savetxt("/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_shear_mod_inv_3_fast/prod_checkpoint_0/tcl_x_c_mech_opt_prod_0.h5.tcl",tcl_file_mech,delimiter=" ", fmt="%s")
radii={}
for i in range(len(all_types)):
    radii[all_types[i]]=1.36

#traj_mech3.convert_to_xyz(particle_radii=radii)


In [ ]:
%%capture
#run multiple copys of sim in parallel
gen_sim_list=[]
gen_sim_param_list=[]

for i in range(5):
    folder_ch="/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_shear_mod_inv_3_fast/eq_checkpoint_"+str(i)+"/"
    folder_pr="/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_shear_mod_inv_3_fast/prod_checkpoint_"+str(i)+"/"
    gen_sim_list.append(run_sim_3)
    run_sim_1_input_c=run_sim_1_input.copy()
    run_sim_1_input_c["t_tot"]=int(1.5*10**7)
    run_sim_1_input_c["k_b_x_c1"]=0.003*3
    run_sim_1_input_c["k_ub_x_c1"]=0.0005*3
    run_sim_1_input_c["k_b_x_hp_c1"]=0
    run_sim_1_input_c["k_ub_x_hp_c1"]=0
    run_sim_1_input_c["new_checkpoint"]=folder_pr
    run_sim_1_input_c["checkpoint_path"]=folder_ch
 
    run_sim_1_input_c["folder"]=folder_pr
    run_sim_1_input_c["add_num_1"]=i
    run_sim_1_input_c["name"]="x_c_mech_opt_prod_"
    gen_sim_param_list.append(run_sim_1_input_c)
    
run_p_dict(gen_sim_list,gen_sim_param_list)

# Simulation run 3

In [ ]:
%%capture
#create equilibration checkpoints


#run multiple copys of sim in parallel
gen_sim_list=[]
gen_sim_param_list=[]

for i in range(5):
    folder_ch="/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_shear_mod_inv_3_slow/eq_checkpoint_"+str(i)+"/"
    folder_pr="/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_shear_mod_inv_3_slow/prod_checkpoint_"+str(i)+"/"
    os.makedirs(folder_ch)
    os.makedirs(folder_pr)
    gen_sim_list.append(run_sim_3)
    run_sim_1_input_c=run_sim_1_input.copy()
    run_sim_1_input_c["t_tot"]=2*10**6
    run_sim_1_input_c["k_b_x_c1"]=0    
    run_sim_1_input_c["k_ub_x_c1"]=0  
    run_sim_1_input_c["k_b_x_hp_c1"]=0
    run_sim_1_input_c["k_ub_x_hp_c1"]=0
    run_sim_1_input_c["new_checkpoint"]=folder_ch
    #run_sim_1_input_c["checkpoint_path"]= "/home/aaron/phd/Nanomotif_Coarse_Graining_1/ML_parameter_opt_1/NCG_X_motif_conrad_1/shear_mod_test_1/checkpoints_15/"
 
    run_sim_1_input_c["folder"]=folder_ch
    run_sim_1_input_c["add_num_1"]=i
    run_sim_1_input_c["name"]="x_c_mech_opt_eq_"
    gen_sim_param_list.append(run_sim_1_input_c)
    
run_p_dict(gen_sim_list,gen_sim_param_list)



In [ ]:
%%capture
#run multiple copys of sim in parallel
gen_sim_list=[]
gen_sim_param_list=[]

for i in range(5):
    folder_ch="/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_shear_mod_inv_3_slow/eq_checkpoint_"+str(i)+"/"
    folder_pr="/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_shear_mod_inv_3_slow/prod_checkpoint_"+str(i)+"/"
    gen_sim_list.append(run_sim_3)
    run_sim_1_input_c=run_sim_1_input.copy()
    run_sim_1_input_c["t_tot"]=int(1.5*10**7)
    run_sim_1_input_c["k_b_x_c1"]=0.003/3
    run_sim_1_input_c["k_ub_x_c1"]=0.0005/3
    run_sim_1_input_c["k_b_x_hp_c1"]=0
    run_sim_1_input_c["k_ub_x_hp_c1"]=0
    run_sim_1_input_c["new_checkpoint"]=folder_pr
    run_sim_1_input_c["checkpoint_path"]=folder_ch
 
    run_sim_1_input_c["folder"]=folder_pr
    run_sim_1_input_c["add_num_1"]=i
    run_sim_1_input_c["name"]="x_c_mech_opt_prod_"
    gen_sim_param_list.append(run_sim_1_input_c)
    
run_p_dict(gen_sim_list,gen_sim_param_list)

In [ ]:
# Simulation run4

In [ ]:
%%capture
#create equilibration checkpoints


#run multiple copys of sim in parallel
gen_sim_list=[]
gen_sim_param_list=[]

for i in range(5):
    folder_ch="/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_shear_mod_5/eq_checkpoint_"+str(i)+"/"
    folder_pr="/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_shear_mod_5/prod_checkpoint_"+str(i)+"/"
    os.makedirs(folder_ch)
    os.makedirs(folder_pr)
    gen_sim_list.append(run_sim_3)
    run_sim_1_input_c=run_sim_1_input.copy()
    run_sim_1_input_c["t_tot"]=2*10**6
    run_sim_1_input_c["k_b_x_c1"]=0    
    run_sim_1_input_c["k_ub_x_c1"]=0  
    run_sim_1_input_c["k_b_x_hp_c1"]=0
    run_sim_1_input_c["k_ub_x_hp_c1"]=0
    run_sim_1_input_c["new_checkpoint"]=folder_ch
    #run_sim_1_input_c["checkpoint_path"]= "/home/aaron/phd/Nanomotif_Coarse_Graining_1/ML_parameter_opt_1/NCG_X_motif_conrad_1/shear_mod_test_1/checkpoints_15/"
 
    run_sim_1_input_c["folder"]=folder_ch
    run_sim_1_input_c["add_num_1"]=i
    run_sim_1_input_c["name"]="x_c_mech_opt_eq_"
    gen_sim_param_list.append(run_sim_1_input_c)
    
run_p_dict(gen_sim_list,gen_sim_param_list)


In [ ]:
%%capture
#run multiple copys of sim in parallel
gen_sim_list=[]
gen_sim_param_list=[]

for i in range(5):
    folder_ch="/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_shear_mod_5/eq_checkpoint_"+str(i)+"/"
    folder_pr="/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_shear_mod_5/prod_checkpoint_"+str(i)+"/"
    gen_sim_list.append(run_sim_3)
    run_sim_1_input_c=run_sim_1_input.copy()
    run_sim_1_input_c["t_tot"]=int(1.5*10**7)
    run_sim_1_input_c["k_ub_x_c1"]=0.000005
    run_sim_1_input_c["new_checkpoint"]=folder_pr
    run_sim_1_input_c["checkpoint_path"]=folder_ch
 
    run_sim_1_input_c["folder"]=folder_pr
    run_sim_1_input_c["add_num_1"]=i
    run_sim_1_input_c["name"]="x_c_mech_opt_prod_"
    gen_sim_param_list.append(run_sim_1_input_c)
    
run_p_dict(gen_sim_list,gen_sim_param_list)

In [ ]:
#run soft 2

In [ ]:
%%capture
#create equilibration checkpoints


#run multiple copys of sim in parallel
gen_sim_list=[]
gen_sim_param_list=[]

for i in range(5):
    folder_ch="/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_shear_mod_soft_2/eq_checkpoint_"+str(i)+"/"
    folder_pr="/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_shear_mod_soft_2/prod_checkpoint_"+str(i)+"/"
    os.makedirs(folder_ch)
    os.makedirs(folder_pr)
    gen_sim_list.append(run_sim_3)
    run_sim_1_input_c=run_sim_1_input.copy()
    run_sim_1_input_c["t_tot"]=2*10**6
    run_sim_1_input_c["k_b_x_c1"]=0    
    run_sim_1_input_c["k_ub_x_c1"]=0  
    run_sim_1_input_c["k_b_x_hp_c1"]=0
    run_sim_1_input_c["k_ub_x_hp_c1"]=0
    run_sim_1_input_c["new_checkpoint"]=folder_ch

    run_sim_1_input_c["k_angle_ds_base"]=10**-24
    #run_sim_1_input_c["k_angle_ds_arm"]=10**-24
    run_sim_1_input_c["k_angle_ds_oppa"]=10**-24
    #run_sim_1_input_c["k_angle_ss_link"]=10**-24
    run_sim_1_input_c["k_angle_ds_fused_link"]=10**-24
    #run_sim_1_input_c["k_angle_ds_hp"]=10**-24



    #run_sim_1_input_c["checkpoint_path"]= "/home/aaron/phd/Nanomotif_Coarse_Graining_1/ML_parameter_opt_1/NCG_X_motif_conrad_1/shear_mod_test_1/checkpoints_15/"
 
    run_sim_1_input_c["folder"]=folder_ch
    run_sim_1_input_c["add_num_1"]=i
    run_sim_1_input_c["name"]="x_c_mech_opt_eq_"
    gen_sim_param_list.append(run_sim_1_input_c)
    
run_p_dict(gen_sim_list,gen_sim_param_list)

In [ ]:
%%capture
#run multiple copys of sim in parallel
gen_sim_list=[]
gen_sim_param_list=[]

for i in range(5):
    folder_ch="/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_shear_mod_soft_2/eq_checkpoint_"+str(i)+"/"
    folder_pr="/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_shear_mod_soft_2/prod_checkpoint_"+str(i)+"/"
    gen_sim_list.append(run_sim_3)
    run_sim_1_input_c=run_sim_1_input.copy()
    run_sim_1_input_c["t_tot"]=int(1.5*10**7)
    run_sim_1_input_c["k_ub_x_c1"]=0.0005
    run_sim_1_input_c["new_checkpoint"]=folder_pr
    run_sim_1_input_c["checkpoint_path"]=folder_ch

    run_sim_1_input_c["k_angle_ds_base"]=10**-24
    #run_sim_1_input_c["k_angle_ds_arm"]=10**-24
    run_sim_1_input_c["k_angle_ds_oppa"]=10**-24
    #run_sim_1_input_c["k_angle_ss_link"]=10**-24
    run_sim_1_input_c["k_angle_ds_fused_link"]=10**-24
    #run_sim_1_input_c["k_angle_ds_hp"]=10**-24

    run_sim_1_input_c["folder"]=folder_pr
    run_sim_1_input_c["add_num_1"]=i
    run_sim_1_input_c["name"]="x_c_mech_opt_prod_"
    gen_sim_param_list.append(run_sim_1_input_c)
    
run_p_dict(gen_sim_list,gen_sim_param_list)

In [ ]:
# run examples of one or two nanomotifs

In [ ]:
#%%capture
#run multiple copys of sim in parallel
gen_sim_list=[]
gen_sim_param_list=[]

for i in range(5):
    #folder_ch="/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_shear_mod_soft_2/eq_checkpoint_"+str(i)+"/"
    folder_pr="/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_shear_mod_soft_2/2_motifs_bound/"
    #os.makedirs(folder_pr)
    gen_sim_list.append(run_sim_3)
    run_sim_1_input_c=run_sim_1_input.copy()
    run_sim_1_input_c["box_size"]=[150,150,150]
    run_sim_1_input_c["num_x_motif_c1"]=1
    run_sim_1_input_c["y_motif_1_rot"]=False

    run_sim_1_input_c["rec_stride"]=100 #recording stride
    run_sim_1_input_c["sim_skin"]=1000 #simulation skin
    
    run_sim_1_input_c["t_tot"]=500000
    
    run_sim_1_input_c["k_b_x_c1"]=0    
    run_sim_1_input_c["k_ub_x_c1"]=0  
    run_sim_1_input_c["k_b_x_hp_c1"]=0
    run_sim_1_input_c["k_ub_x_hp_c1"]=0
    #run_sim_1_input_c["new_checkpoint"]=folder_pr
    #run_sim_1_input_c["checkpoint_path"]=folder_ch

    run_sim_1_input_c["k_angle_ds_base"]=10**-24
    #run_sim_1_input_c["k_angle_ds_arm"]=10**-24
    run_sim_1_input_c["k_angle_ds_oppa"]=10**-24
    #run_sim_1_input_c["k_angle_ss_link"]=10**-24
    run_sim_1_input_c["k_angle_ds_fused_link"]=10**-24
    #run_sim_1_input_c["k_angle_ds_hp"]=10**-24

    #run_sim_1_input_c["use_x_c1_HP_template"]=True
    run_sim_1_input_c["checkpoint_path"]="/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_shear_mod_soft_2/two_fused_conrad_x_motifs_checkpoint/"

    run_sim_1_input_c["folder"]=folder_pr
    run_sim_1_input_c["add_num_1"]=i
    run_sim_1_input_c["name"]="x_c_mech_opt_angle_test_bound_"
    gen_sim_param_list.append(run_sim_1_input_c)
    
run_p_dict(gen_sim_list,gen_sim_param_list)

In [ ]:
# Run simulations for videos

In [ ]:
#run multiple copys of sim in parallel
gen_sim_list=[]
gen_sim_param_list=[]

for i in range(1):
    gen_sim_list.append(run_sim_3)
    run_sim_1_input_c=run_sim_1_input.copy()
    run_sim_1_input_c["x_motif_c1_ini"]=[np.asarray([[0,0,0]])]
    run_sim_1_input_c["box_size"]=[150,150,150]
    run_sim_1_input_c["num_x_motif_c1"]=1
    run_sim_1_input_c["x_motif_c1_rot"]=False
    run_sim_1_input_c["t_tot"]=1*10**5
    run_sim_1_input_c["k_b_x_c1"]=0    
    run_sim_1_input_c["k_ub_x_c1"]=0  
    run_sim_1_input_c["k_b_x_hp_c1"]=0
    run_sim_1_input_c["k_ub_x_hp_c1"]=0
    run_sim_1_input_c["rec_stride"]=10 #recording stride
    run_sim_1_input_c["sim_skin"]=1000 #simulation skin
    #run_sim_1_input_c["new_checkpoint"]="/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_shear_mod_2/eq_checkpoint_"+str(i)+"/"
    #run_sim_1_input_c["checkpoint_path"]= "/home/aaron/phd/Nanomotif_Coarse_Graining_1/ML_parameter_opt_1/NCG_X_motif_conrad_1/shear_mod_test_1/checkpoints_15/"
 
    run_sim_1_input_c["folder"]="/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_video_1_motif/"
    run_sim_1_input_c["add_num_1"]=i
    run_sim_1_input_c["name"]="x_c_mech_opt_eq_"
    gen_sim_param_list.append(run_sim_1_input_c)
    
run_p_dict(gen_sim_list,gen_sim_param_list)


In [ ]:
#run multiple copys of sim in parallel
gen_sim_list=[]
gen_sim_param_list=[]

for i in range(1):
    gen_sim_list.append(run_sim_3)
    run_sim_1_input_c=run_sim_1_input.copy()
    #run_sim_1_input_c["x_motif_c1_ini"]=[np.asarray([[0,0,0]])]
    #run_sim_1_input_c["box_size"]=[150,150,150]
    #run_sim_1_input_c["num_x_motif_c1"]=1
    #run_sim_1_input_c["x_motif_c1_rot"]=False
    run_sim_1_input_c["t_tot"]=1*10**5
    #run_sim_1_input_c["k_b_x_c1"]=0    
    #run_sim_1_input_c["k_ub_x_c1"]=0  
    #run_sim_1_input_c["k_b_x_hp_c1"]=0
    #run_sim_1_input_c["k_ub_x_hp_c1"]=0
    run_sim_1_input_c["rec_stride"]=10 #recording stride
    #run_sim_1_input_c["sim_skin"]=1000 #simulation skin
    #run_sim_1_input_c["new_checkpoint"]="/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_shear_mod_2/eq_checkpoint_"+str(i)+"/"
    run_sim_1_input_c["checkpoint_path"]= "/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_video_network_shear_mod_2/"
 
    run_sim_1_input_c["folder"]="/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_video_network_shear_mod_2/"
    run_sim_1_input_c["add_num_1"]=i
    run_sim_1_input_c["name"]="x_c_mech_opt_eq_"
    gen_sim_param_list.append(run_sim_1_input_c)
    
run_p_dict(gen_sim_list,gen_sim_param_list)

In [ ]:
traj_mech=readdy.Trajectory("/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_video_network_shear_mod_2/x_c_mech_opt_eq_0.h5")

times_mech,types_mech,ids_mech,part_positions_mech=traj_mech.read_observable_particles()
time_pt_mech, counts_pt_mech = traj_mech.read_observable_number_of_particles()
#part_pos_eq=part_positions

time_e_mech, energy_mech=traj_mech.read_observable_energy()

time_obs_top_mech,obs_top_mech=traj_mech.read_observable_topologies()

In [ ]:
radii={}
for i in range(len(all_types)):
    radii[all_types[i]]=1.36

traj_mech.convert_to_xyz(particle_radii=radii)

In [ ]:
tcl_file_mech=tcl_file_gen_3(traj_file="x_c_mech_opt_eq_0.h5.xyz",box_dim_s="69.718 69.718 69.718")

np.savetxt("/mnt/d/D_phd/D_Nanomotif_Coarse_Graining_1/D_ML_parameter_opt_1/D_NCG_X_motif_conrad_1/D_video_network_shear_mod_2/tcl_x_c_mech_opt_eq_0.h5.tcl",tcl_file_mech,delimiter=" ", fmt="%s")